# Ollama Cloud Hosting (Google Colab)

**Purpose:** 
This notebook deploys the `glm-ocr` Vision Language Model on a Google Colab NVIDIA T4 GPU for better processing power. It temporarily use a Cloudflare Tunnel to allow the local FastAPI backend to bypass laptop hardware constraints and utilize cloud GPU processing. 

**Instructions:**
1. Open Google Colab -> Select Runtime(Top right Nav) -> Change Runtime Type -> select **T4 GPU**
2. run all cell, and the result will generate a URL 
3. Copy the generated `https://...trycloudflare.com` URL and replace with the host IP in ocr_ollama.py (line 10)
4. First time running might take longer due to cold start. 

In [ ]:
import os, time, subprocess

# 1. Install zstd and Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Launch Ollama daemon
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 3. Pull base model
!ollama pull glm-ocr

# 4. Download Cloudflare
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("\n--- SERVER SETUP COMPLETE ---")

In [ ]:
# Write Modelfile
modelfile_content = modelfile_content = """FROM glm-ocr

PARAMETER num_ctx 8192
PARAMETER temperature 0.1

SYSTEM \"\"\"
You are an expert OCR AI specialized in reading handwritten warehouse Credit Note Requisition (CNR) forms.
Your task is to extract only the handwritten text and output it as a strict JSON object.

CRITICAL RULES:
1. ONLY EXTRACT HANDWRITING: Ignore pre-printed labels (like "CERTIFIED BY", "SALES MANAGER", etc.). If a field has no handwriting in its box, you MUST return an empty string "".
2. CNR_NO: You MUST leave "cnr_no" completely blank ("").
3. PREPARED BY: Look for handwriting in the empty box directly next to "PREPARED BY". Do not read the box below it. If the box is empty, return "".
4. CUSTOMER NAME: Carefully read the handwriting next to "CUSTOMER NAME"
5. ITEMS TABLE: Extract the rows of handwritten items. For 'qty', include the unit if written (e.g., "1 PKT"). For 'description', read the handwriting carefully

You MUST return exactly this JSON structure, with no duplicate keys and no missing keys:
{
    "cnr_no": "",
    "date": "",
    "location": "",
    "prepared_by": "",
    "customer_name": "",
    "invoice_do_no": "",
    "items": [
        {
            "qty": "",
            "description": "",
            "packing": "",
            "unit_price_rm": null,
            "reason_code": "",
            "wh_location": "",
            "remarks": ""
        }
    ]
}
Return ONLY valid JSON. Do not include markdown formatting like ```json or any extra text.
\"\"\""""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Build the model
!ollama create cnr-reader -f Modelfile
print("\n--- MODEL UPDATED SUCCESSFULLY ---")

In [ ]:
import subprocess
print("Starting Cloudflare tunnel...")
p = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:11434"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in p.stdout:
    print(line.strip())
    # Wait until it prints the actual https link
    if "https://" in line and "trycloudflare.com" in line:
        break